In [ ]:
! git clone https://github.com/fssdfhsdfsdk/CUDA-by-Example-source-code-for-the-book-s-examples-.git

Cloning into 'CUDA-by-Example-source-code-for-the-book-s-examples-'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 119 (delta 0), reused 2 (delta 0), pack-reused 115 (from 1)
Receiving objects: 100% (119/119), 1.09 MiB | 3.42 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [2]:
%cd /content/CUDA-by-Example-source-code-for-the-book-s-examples- 

/content/CUDA-by-Example-source-code-for-the-book-s-examples-


## add_loop_cpu.cu

In [50]:
%%writefile add_loop_cpu.cu

#include "common/book.h"
#define N 10

void add(int *a, int *b, int *c) {
    int tid = 0;
    while(tid < N) {
        c[tid] = a[tid] + b[tid];
        tid ++;  // 只有一个cpu，每次只加1
    }
}


int main(void) {
    int a[N], b[N], c[N];

    for(int i=0; i < N; i++) {
        a[i] = -i;
        b[i] = i * i;
    }



    add(a, b, c);

    for(int i=0; i < N; i++) {
        printf("%d ", c[i]);
    } 
    printf("\n");

    return 0;
}



Overwriting add_loop_cpu.cu


In [67]:
! nvcc add_loop_cpu.cu -arch=sm_75 -o hello && date; ./hello; date

Fri Mar  6 04:12:12 AM UTC 2026
0 0 2 6 12 20 30 42 56 72 
Fri Mar  6 04:12:12 AM UTC 2026


这個 **Segmentation fault (段错误)** 是因为你把太大的数组放进了 **栈 (Stack)** 里。


核心原因
在 C 语言中，函数内部定义的 `int a[N]` 属于局部变量，存储在栈空间。

-   你的 `N` 是 `1,000,000`。
- 三个 `int` 数组总共占用：12MB
-   大多数 Linux 系统的默认栈限制（ulimit）只有 **8MB**。
-   一旦程序运行，内存需求超过栈容量，直接崩掉。

ulimit -s unlimited;

## add_loop_cpu_pthread.cu

In [54]:
%%writefile add_loop_cpu_pthread.cu

#include "common/book.h"
#include <iostream>
#include <pthread.h>
#include <string>
#define N 10

// 1. 定义参数结构体
struct ThreadArgs {
    int * a;
    int * b;
    int * c;
    int tid;
};

void* add(void *arg) {
    ThreadArgs* data = static_cast<ThreadArgs*>(arg);

    int tid = data->tid;
    while(tid < N) {
        data->c[tid] = data->b[tid] + data->a[tid];
        tid ++;
    }
    delete data;
    return nullptr;
}




int main(void) {
    int a[N], b[N], c[N];

    const int NUM_THREADS = 2;
    pthread_t threads[NUM_THREADS]; // 线程句柄数组

    for(int i=0; i < N; i++) {
        a[i] = -i;
        b[i] = i * i;
    }



    for (long i = 0; i < NUM_THREADS; i++) {
        ThreadArgs* args = new ThreadArgs();
        args->a = a;
        args->b = b;
        args->c = c;
        args->tid = i; 
        // 3. 创建线程：句柄, 属性(通常NULL), 函数名, 参数
        int rc = pthread_create(&threads[i], NULL, add, (void*)args);
        
        if (rc) {
            std::cerr << "错误：无法创建线程," << rc << std::endl;
            exit(-1);
        }
    }

    for (int i = 0; i < NUM_THREADS; i++) {
        void* status;
        pthread_join(threads[i], &status);
        std::cout << "线程 " << i << " 已结束，返回码: " << (long)status << std::endl;
    }


    for(int i=0; i < N; i++) {
        printf("%d ", c[i]);
    }


    return 0;
}



Overwriting add_loop_cpu_pthread.cu


In [55]:
! nvcc add_loop_cpu_pthread.cu -arch=sm_75 -o hello && ./hello

线程 0 已结束，返回码: 0
线程 1 已结束，返回码: 0
0 0 2 6 12 20 30 42 56 72 

## add_loop_gpu.cu



In [56]:
%%writefile add_loop_gpu.cu
#include "common/book.h"

#define N 100000


__global__ void add(int *a, int *b, int *c) {
    int tid = blockIdx.x;
    while(tid < N) {
        c[tid] = a[tid] + b[tid];
        tid += N;
    }
}


int main(void) {
    int a[N], b[N], c[N];
    int * dev_a, * dev_b, * dev_c;

    for(int i=0; i < N; i++) {
        a[i] = -i;
        b[i] = i * i;
    }

    HANDLE_ERROR( cudaMalloc((void **)&dev_a, N * sizeof(int)));
    HANDLE_ERROR( cudaMalloc((void **)&dev_b, N * sizeof(int)));
    HANDLE_ERROR( cudaMalloc((void **)&dev_c, N * sizeof(int)));

    HANDLE_ERROR( cudaMemcpy(dev_a, a, N * sizeof(int), cudaMemcpyHostToDevice));
    HANDLE_ERROR( cudaMemcpy(dev_b, b, N * sizeof(int), cudaMemcpyHostToDevice));

    add<<<N,1>>>(dev_a, dev_b, dev_c);

    HANDLE_ERROR( cudaMemcpy(c, dev_c, N * sizeof(int), cudaMemcpyDeviceToHost));

/*
    for(int i=0; i < N; i++) {
        printf("%d ", c[i]);
    }
*/

    cudaFree(dev_a);
    cudaFree(dev_b);
    cudaFree(dev_c);

    return 0;
}

Overwriting add_loop_gpu.cu


测试运行：
 - 100000（10w个block），正常运行

In [58]:
! nvcc add_loop_gpu.cu -arch=sm_75 -o hello && date; ./hello; date;date

Fri Mar  6 04:05:43 AM UTC 2026
Fri Mar  6 04:05:44 AM UTC 2026
Fri Mar  6 04:05:44 AM UTC 2026


## julia_cpu.cu

In [59]:
%%writefile julia_cpu.cu
#include "common/book.h"
#include "common/cpu_bitmap.h"

#define DIM 1000

struct cuComplex {
    float   r;
    float   i;
    cuComplex( float a, float b ) : r(a), i(b)  {}
    float magnitude2( void ) { return r * r + i * i; }
    cuComplex operator*(const cuComplex& a) {
        return cuComplex(r*a.r - i*a.i, i*a.r + r*a.i);
    }
    cuComplex operator+(const cuComplex& a) {
        return cuComplex(r+a.r, i+a.i);
    }
};

int julia( int x, int y ) { 
    const float scale = 1.5;
    float jx = scale * (float)(DIM/2 - x)/(DIM/2);
    float jy = scale * (float)(DIM/2 - y)/(DIM/2);

    cuComplex c(-0.8, 0.156);
    cuComplex a(jx, jy);

    int i = 0;
    for (i=0; i<200; i++) {
        a = a * a + c;
        if (a.magnitude2() > 1000)
            return 0;
    }

    return 1;
}

void kernel( unsigned char *ptr ){
    for (int y=0; y<DIM; y++) {
        for (int x=0; x<DIM; x++) {
            int offset = x + y * DIM;

            int juliaValue = julia( x, y );
            ptr[offset*4 + 0] = 255 * juliaValue;
            ptr[offset*4 + 1] = 0;
            ptr[offset*4 + 2] = 0;
            ptr[offset*4 + 3] = 255;
        }
    }
 }

int main( void ) {
    CPUBitmap bitmap( DIM, DIM );
    unsigned char *ptr = bitmap.get_ptr();

    kernel( ptr );

    bitmap.display_and_exit();
}



Writing julia_cpu.cu


In [68]:
! nvcc julia_cpu.cu -arch=sm_75 -o hello -L/usr/lib/x86_64-linux-gnu -lglut -lGLU -lGL && date; ./hello; date;date

common/cpu_bitmap.h(49): warning #2464-D: conversion from a string literal to "char *" is deprecated
          char* dummy = "";
                        ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

common/cpu_bitmap.h(49): warning #2464-D: conversion from a string literal to "char *" is deprecated
          char* dummy = "";
                        ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

/usr/bin/ld: /usr/lib/x86_64-linux-gnu/libGL.so: undefined reference to `_glapi_tls_Current'
collect2: error: ld returned 1 exit status
/bin/bash: line 1: ./hello: No such file or directory
Fri Mar  6 04:14:22 AM UTC 2026
Fri Mar  6 04:14:22 AM UTC 2026


In [62]:
! sudo apt-get install freeglut3-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  freeglut3 libegl-dev libgl-dev libgl1-mesa-dev libgles-dev libgles1
  libglu1-mesa libglu1-mesa-dev libglvnd-core-dev libglvnd-dev libglx-dev
  libopengl-dev libxt-dev
Suggested packages:
  libxt-doc
The following NEW packages will be installed:
  freeglut3 freeglut3-dev libegl-dev libgl-dev libgl1-mesa-dev libgles-dev
  libgles1 libglu1-mesa libglu1-mesa-dev libglvnd-core-dev libglvnd-dev
  libglx-dev libopengl-dev libxt-dev
0 upgraded, 14 newly installed, 0 to remove and 37 not upgraded.
Need to get 1,192 kB of archives.
After this operation, 6,439 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 freeglut3 amd64 2.8.1-6 [74.0 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglx-dev amd64 1.4.0-1 [14.1 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgl-de